In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_groq import ChatGroq
api_key=os.getenv("GROQ_API_KEY")
llm=ChatGroq(api_key=api_key,model="openai/gpt-oss-safeguard-20b")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.15'}}, profile={'name': 'Safety GPT OSS 20B', 'release_date': '2025-03-05', 'last_updated': '2025-03-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001EEAB4158A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EEAB4157B0>, model_name='openai/gpt-oss-safeguard-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [3]:
from langchain_core.messages import (
    AIMessage,
    SystemMessage,
    HumanMessage
)

In [4]:
speech="""
Modern speech AI systems use deep neural network (DNN) models trained on massive datasets. 
Over time, the size of speech AI models has grown so much that training such models can take weeks of 
intensive compute time, even when using deep learning frameworks, such as PyTorch, TensorFlow, and MXNet, 
on high-performance GPUs.
Many enterprises have to customize speech and translation AI models to achieve the desired 
multilingual accuracy for their specific conversational applications. However, customizing speech AI models 
from scratch usually requires large training datasets and AI expertise.
For speech AI skills, companies have always had to choose between accuracy and real-time performance. 
For example, they can’t ask a question and then wait several seconds for a response. 
In addition, they don’t want their conversational AI applications to misinterpret or produce gibberish.
With NVIDIA Riva, companies can achieve world-class accuracy and run their speech and translation 
AI pipelines in real time—under a few milliseconds. Riva offers SOTA pretrained models on NGC that could 
be fine-tuned with NVIDIA NeMo to achieve world-class accuracy, and optimized skills for real-time performance.
"""

In [5]:
chat_messages=[
    SystemMessage(content="Yor are an expert with summarizing the speech"),
    HumanMessage(content=f"Please provide a short and concise summary of the following speech: \n Text:{speech}")
]

In [6]:
llm.get_num_tokens(speech)

c:\Users\anish\Documents\LANGCHAIN\venv\lib\site-packages\langchain_core\language_models\base.py:463: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
c:\Users\anish\Documents\LANGCHAIN\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


263

## Pormpt template for summarization

In [7]:
from langchain_core.prompts import PromptTemplate

generic_template="""
Write a summary of the following speech:
Speech:{speech}
Translate the precise summary to {language}
"""

prompt=PromptTemplate(
    input_variables=['speech','language'],
    template=generic_template
)

prompt

PromptTemplate(input_variables=['language', 'speech'], input_types={}, partial_variables={}, template='\nWrite a summary of the following speech:\nSpeech:{speech}\nTranslate the precise summary to {language}\n')

In [8]:
complete_prompt=prompt.format(speech=speech,language="Hindi")
complete_prompt

'\nWrite a summary of the following speech:\nSpeech:\nModern speech AI systems use deep neural network (DNN) models trained on massive datasets. \nOver time, the size of speech AI models has grown so much that training such models can take weeks of \nintensive compute time, even when using deep learning frameworks, such as PyTorch, TensorFlow, and MXNet, \non high-performance GPUs.\nMany enterprises have to customize speech and translation AI models to achieve the desired \nmultilingual accuracy for their specific conversational applications. However, customizing speech AI models \nfrom scratch usually requires large training datasets and AI expertise.\nFor speech AI skills, companies have always had to choose between accuracy and real-time performance. \nFor example, they can’t ask a question and then wait several seconds for a response. \nIn addition, they don’t want their conversational AI applications to misinterpret or produce gibberish.\nWith NVIDIA Riva, companies can achieve wo

In [9]:
llm.get_num_tokens(complete_prompt)

285

In [10]:
from langchain_classic.chains import LLMChain
llm_chain=LLMChain(llm=llm,prompt=prompt)
summary=llm_chain.invoke({"speech":speech,"language":'Hindi'})["text"]
summary

C:\Users\anish\AppData\Local\Temp\ipykernel_6368\1146183601.py:2: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain=LLMChain(llm=llm,prompt=prompt)


'**English Summary**\n\nModern speech‑AI relies on huge deep‑neural‑network models that can take weeks to train, even on powerful GPUs. Enterprises often need to fine‑tune these models for specific multilingual use‑cases, but doing so from scratch demands large datasets and deep‑learning expertise. A key challenge is balancing high accuracy with real‑time responsiveness; users cannot afford a multi‑second lag or garbled output. NVIDIA\u202fRiva addresses this by offering state‑of‑the‑art pretrained models on NGC that can be fine‑tuned with NVIDIA\u202fNeMo, delivering world‑class accuracy while enabling sub‑millisecond real‑time inference for speech and translation pipelines.\n\n**Hindi Translation**\n\nआधुनिक स्पीच‑एआई गहन न्यूरल नेटवर्क (DNN) मॉडलों पर आधारित है, जिन्हें प्रशिक्षित करने में साप्ताहिक उच्च‑प्रदर्शन GPUs पर भी हफ्तों का गहन कंप्यूट समय लग सकता है। कंपनियों को अक्सर अपने विशिष्ट बहुभाषी अनुप्रयोगों के लिए इन मॉडलों को अनुकूलित करने की आवश्यकता होती है, लेकिन ऐसा कार्य ब

## StuffDocumentChain Text summarizer

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader("266_Research_Paper.pdf")
docs=loader.load()
docs


C:\Users\anish\AppData\Local\Temp\ipykernel_6368\4089185167.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


[Document(metadata={'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': 'D:20170629124415', 'author': 'ARYAAN', 'moddate': 'D:20170629124415', 'source': '266_Research_Paper.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='www.epitomejournals.com  Impact Factor  3.656, Vol. III, Issue VI, June 2017, ISSN:  2395-6968 \n \n145 KV        Dr. Pramod Ambadasrao Pawar, Editor-in-Chief ©EIJMR, All rights reserved. \n \n \nGlobal Warming and Climate Change an Overview :  \nAlarmism or Skepticism \n \nVishal Kumar \nAssistant Professor in Geography \nGovt. College Sarkaghat Distt. Mandi (Himachal Pradesh) \nE-mail: vt220101@gmail.com \n \n \nABSTRACT \nGlobal warming refers to the increase in the earth’s average surface temperature since the \nIndustrial Revolution, primarily due to the emission of greenhouse gases from the burning of \nfossil fuels and land use change. Global surface temperature has increased by about \n0.74 

In [12]:
template="""
Write a concise and short summary of the following speech,
Speech:{text}
"""

prompt=PromptTemplate(input_variables=["text"],template=template)

In [15]:
from langchain_classic.chains.summarize import load_summarize_chain

In [16]:
"""
This causes error due to : With chain_type="stuff", all PDF content is combined into one request:

Requested: 8621 tokens
Your limit: 8000 tokens, hence we got the error

chain=load_summarize_chain(llm,chain_type="stuff",verbose=True,prompt=prompt)
output=chain.invoke(docs)
output
"""

'\nThis causes error due to : With chain_type="stuff", all PDF content is combined into one request:\n\nRequested: 8621 tokens\nYour limit: 8000 tokens, hence we got the error\n\nchain=load_summarize_chain(llm,chain_type="stuff",verbose=True,prompt=prompt)\noutput=chain.invoke(docs)\noutput\n'

In [17]:
## Map reduce to summarize the document

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [19]:
docs

[Document(metadata={'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': 'D:20170629124415', 'author': 'ARYAAN', 'moddate': 'D:20170629124415', 'source': '266_Research_Paper.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='www.epitomejournals.com  Impact Factor  3.656, Vol. III, Issue VI, June 2017, ISSN:  2395-6968 \n \n145 KV        Dr. Pramod Ambadasrao Pawar, Editor-in-Chief ©EIJMR, All rights reserved. \n \n \nGlobal Warming and Climate Change an Overview :  \nAlarmism or Skepticism \n \nVishal Kumar \nAssistant Professor in Geography \nGovt. College Sarkaghat Distt. Mandi (Himachal Pradesh) \nE-mail: vt220101@gmail.com \n \n \nABSTRACT \nGlobal warming refers to the increase in the earth’s average surface temperature since the \nIndustrial Revolution, primarily due to the emission of greenhouse gases from the burning of \nfossil fuels and land use change. Global surface temperature has increased by about \n0.74 

In [20]:
final_doc=RecursiveCharacterTextSplitter(chunk_size=2000,chunk_overlap=100).split_documents(docs)
final_doc

[Document(metadata={'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': 'D:20170629124415', 'author': 'ARYAAN', 'moddate': 'D:20170629124415', 'source': '266_Research_Paper.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='www.epitomejournals.com  Impact Factor  3.656, Vol. III, Issue VI, June 2017, ISSN:  2395-6968 \n \n145 KV        Dr. Pramod Ambadasrao Pawar, Editor-in-Chief ©EIJMR, All rights reserved. \n \n \nGlobal Warming and Climate Change an Overview :  \nAlarmism or Skepticism \n \nVishal Kumar \nAssistant Professor in Geography \nGovt. College Sarkaghat Distt. Mandi (Himachal Pradesh) \nE-mail: vt220101@gmail.com \n \n \nABSTRACT \nGlobal warming refers to the increase in the earth’s average surface temperature since the \nIndustrial Revolution, primarily due to the emission of greenhouse gases from the burning of \nfossil fuels and land use change. Global surface temperature has increased by about \n0.74 

In [21]:
len(final_doc)

27

In [22]:
chunk_prompt="""
PLease summarize this below speech:
Speech:'{text}'
Summary: 
"""

map_prompt_template=PromptTemplate(input_variables=['text'],
                                   template=chunk_prompt)

In [23]:
final_prompt="""
provide the final summary of the entire speech with these important points.
Add a Motivational Title, start with precise summary with an introduction and provide the summary in number points for the speech. 
Speech :{text}
"""

final_prompt_template=PromptTemplate(input_variables=['text'],
                                     template=final_prompt)

In [24]:
summary_chain=load_summarize_chain(llm=llm,chain_type="map_reduce",verbose=True,map_prompt=map_prompt_template,combine_prompt=final_prompt_template)


In [25]:
output = summary_chain.invoke({
    "input_documents": final_doc
})

print(output)



> Entering new MapReduceDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

PLease summarize this below speech:
Speech:'www.epitomejournals.com  Impact Factor  3.656, Vol. III, Issue VI, June 2017, ISSN:  2395-6968 
 
145 KV        Dr. Pramod Ambadasrao Pawar, Editor-in-Chief ©EIJMR, All rights reserved. 
 
 
Global Warming and Climate Change an Overview :  
Alarmism or Skepticism 
 
Vishal Kumar 
Assistant Professor in Geography 
Govt. College Sarkaghat Distt. Mandi (Himachal Pradesh) 
E-mail: vt220101@gmail.com 
 
 
ABSTRACT 
Global warming refers to the increase in the earth’s average surface temperature since the 
Industrial Revolution, primarily due to the emission of greenhouse gases from the burning of 
fossil fuels and land use change. Global surface temperature has increased by about 
0.74 ± 0.18 °C (1.33 ± 0.32 °F) between the start and the end of the 20th century.The rate of 
warming over the last half of that period was almost double than 

In [27]:
output["output_text"]

'**Motivational Title**  \n**“Turning the Tide: Blueprint for a Climate‑Proof Tomorrow”**\n\n---\n\n### Precise Summary (Introduction)\n\nDr.\u202fPramod Ambadasrao\u202fPawar’s 2017 address stitches a clear, science‑backed narrative of the warming planet, its cascading effects on ecosystems, health, and economies, and the urgent need for collective, sector‑wide action. It calls for a rapid transition to a zero‑carbon future, a circular economy, and resilient policies that embed sustainability into every layer of society. The speech is a rallying cry that “prevention beats cure” and that the window for meaningful mitigation is now.\n\n---\n\n### Numbered Take‑aways\n\n1. **The Science is Settled**  \n   * Global temperature ↑\u202f0.74\u202f°C in the 20th\u202fcentury, accelerating in the 2nd half.  \n   * Earth’s energy balance is skewed: more solar energy trapped by GHGs than radiated.\n\n2. **Primary Drivers of Warming**  \n   * Fossil‑fuel combustion (~70\u202f% of rise since 1970)